In [ ]:
%pip install pyswarms
%pip install lime


In [ ]:
!pip install tensorflow-addons --quiet
print("tensorflow-addons installed successfully.")

ERROR: Could not find a version that satisfies the requirement tensorflow-addons (from versions: none)
ERROR: No matching distribution found for tensorflow-addons
tensorflow-addons installed successfully.


In [ ]:
!pip install tensorflow-extra --quiet
print("tensorflow-extra installed successfully.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 620.7/620.7 MB 727.1 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.5/57.5 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.5/24.5 MB 69.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 117.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 129.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 225.0/225.0 kB 23.2 MB/s eta 0:00:00
tensorflow-extra installed successfully.


In [ ]:
!pip install tf-keras --quiet
import tensorflow_extra
print("tensorflow_extra imported successfully.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 30.1 MB/s eta 0:00:00
tensorflow_extra imported successfully.


In [ ]:
# =======================
# 0. BASIC SETUP
# =======================
!pip install pyswarms --quiet
!pip install lime --quiet
!pip install opencv-python --quiet

import os, random
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import matplotlib.pyplot as plt
import cv2

from lime import lime_image
from skimage.segmentation import mark_boundaries

from google.colab import drive
drive.mount('/content/drive')

# === PATHS ===
BASE_PATH = "/content/drive/MyDrive/LungNodule_Dataset/lungnodulenewimagedataset"
train_dir = os.path.join(BASE_PATH, 'train_set')
val_dir   = os.path.join(BASE_PATH, 'validation_set')
test_dir  = os.path.join(BASE_PATH, 'test_set')

IMG_SIZE = (224, 224)
NUM_CLIENTS = 5
LOCAL_BATCH_SIZE = 32
COMM_ROUNDS = 10

# Enable GPU growth (optional)
gpus = tf.config.list_physical_devices('GPU')
for g in gpus:
    try:
        tf.config.experimental.set_memory_growth(g, True)
    except:
        pass

print("Base path:", BASE_PATH)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.1/104.1 kB 3.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 275.7/275.7 kB 11.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done


ValueError: mount failed

In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# =======================
# 1. LOAD DATA (CENTRAL)
# =======================
train_datagen = ImageDataGenerator(
    rescale=1./255.,
    horizontal_flip=True,
    vertical_flip=True,
    rotation_range=10,
    width_shift_range=0.05,
    height_shift_range=0.05,
)

val_datagen = ImageDataGenerator(rescale=1./255.)
test_datagen = ImageDataGenerator(rescale=1./255.)

# Make a generator just to materialize everything once
train_gen_full = train_datagen.flow_from_directory(
    train_dir,
    target_size=IMG_SIZE,
    batch_size=64,
    class_mode='binary',
    shuffle=True
)

val_gen = val_datagen.flow_from_directory(
    val_dir,
    target_size=IMG_SIZE,
    batch_size=32,
    class_mode='binary',
    shuffle=False
)

test_gen = test_datagen.flow_from_directory(
    test_dir,
    target_size=IMG_SIZE,
    batch_size=32,
    class_mode='binary',
    shuffle=False
)

print("Class indices:", train_gen_full.class_indices)

# ---- Materialize entire train set (once) ----
train_images, train_labels = [], []
for i in range(len(train_gen_full)):
    x, y = next(train_gen_full)
    train_images.append(x)
    train_labels.append(y)
    if (i+1) % 10 == 0:
        print(f"Loaded {i+1}/{len(train_gen_full)} train batches")

train_images = np.concatenate(train_images, axis=0)
train_labels = np.concatenate(train_labels, axis=0)
print("Train data:", train_images.shape, train_labels.shape)


Found 5187 images belonging to 2 classes.
Found 1287 images belonging to 2 classes.
Found 1600 images belonging to 2 classes.
Class indices: {'nodule': 0, 'non nodule': 1}
Loaded 10/82 train batches
Loaded 20/82 train batches
Loaded 30/82 train batches
Loaded 40/82 train batches
Loaded 50/82 train batches
Loaded 60/82 train batches
Loaded 70/82 train batches
Loaded 80/82 train batches
Train data: (5187, 224, 224, 3) (5187,)


In [ ]:
# =======================
# 2. CREATE CLIENT DATASETS
# =======================
def create_client_datasets(images, labels, num_clients):
    idx = np.arange(len(images))
    np.random.shuffle(idx)
    images, labels = images[idx], labels[idx]
    splits = np.array_split(np.arange(len(images)), num_clients)
    client_data = []
    for s in splits:
        client_data.append((images[s], labels[s]))
    return client_data

client_datasets = create_client_datasets(train_images, train_labels, NUM_CLIENTS)
client_sizes = [len(c[0]) for c in client_datasets]
print("Client sizes:", client_sizes)

def make_tf_dataset(images, labels, batch_size):
    ds = tf.data.Dataset.from_tensor_slices((images, labels))
    ds = ds.shuffle(buffer_size=len(images)).batch(batch_size).prefetch(tf.data.AUTOTUNE)
    return ds

client_tf_datasets = [
    make_tf_dataset(imgs, lbls, LOCAL_BATCH_SIZE)
    for (imgs, lbls) in client_datasets
]


Client sizes: [1038, 1038, 1037, 1037, 1037]


In [ ]:
# =======================
# 3. LUNG-ATTNET STYLE MODEL (LGAM)
# =======================
def spatial_attention_block(F):
    avg_pool = layers.Lambda(lambda x: tf.reduce_mean(x, axis=-1, keepdims=True))(F)
    max_pool = layers.Lambda(lambda x: tf.reduce_max(x, axis=-1, keepdims=True))(F)
    concat = layers.Concatenate(axis=-1)([avg_pool, max_pool])
    x = layers.Conv2D(1, (1,1), activation='sigmoid', padding='same')(concat)
    return layers.Multiply()([F, x])

def self_attention_block(F):
    C = F.shape[-1]
    C_ = C // 8 if C is not None and C >= 8 else 4
    f = layers.Conv2D(C_, 1, padding='same')(F)
    g = layers.Conv2D(C_, 1, padding='same')(F)
    h = layers.Conv2D(C_, 1, padding='same')(F)

    f_flat = layers.Reshape((-1, C_))(f)
    g_flat = layers.Reshape((-1, C_))(g)
    h_flat = layers.Reshape((-1, C_))(h)

    s = layers.Lambda(lambda args: tf.matmul(args[0], args[1], transpose_b=True))([f_flat, g_flat])
    beta = layers.Softmax(axis=-1)(s)
    o = layers.Lambda(lambda args: tf.matmul(args[0], args[1]))([beta, h_flat])
    # Re-apply the corrected reshape target from previous debugging session
    o = layers.Reshape((F.shape[1], F.shape[2], C_))(o)
    o = layers.Conv2D(C, 1, activation='relu', padding='same')(o)
    return o

def channel_attention_block(F):
    gap = layers.GlobalMaxPooling2D()(F)
    x = layers.Reshape((1,1,F.shape[-1]))(gap)
    x = layers.Conv2D(F.shape[-1]//2, 1, activation='relu', padding='same')(x)
    x = layers.Conv2D(F.shape[-1], 1, activation='sigmoid', padding='same')(x)
    return layers.Multiply()([F, x])

def LGAM_block(F):
    F_spatial = spatial_attention_block(F)
    F_self = self_attention_block(F)
    F_agg = layers.Add()([F_spatial, F_self])
    F_ch = channel_attention_block(F_agg)
    return F_ch

def build_lung_attnet(input_shape=(224,224,3), num_classes=1):
    inputs = layers.Input(shape=input_shape)

    x = layers.Conv2D(32, 3, padding='same', activation='relu')(inputs)
    x = layers.MaxPooling2D(2)(x)

    x = layers.Conv2D(64, 3, padding='same', activation='relu')(x)
    x = layers.MaxPooling2D(2)(x)

    x = layers.Conv2D(128, 3, padding='same', activation='relu')(x)
    x = layers.MaxPooling2D(2)(x)

    x = layers.Conv2D(256, 3, padding='same', activation='relu')(x)
    x = layers.MaxPooling2D(2)(x)

    x = LGAM_block(x)

    x = layers.Conv2D(256, 3, padding='same', activation='relu')(x)
    x = layers.Flatten()(x)
    x = layers.Dense(128, activation='relu')(x)
    x = layers.Dropout(0.4)(x)
    x = layers.Dense(256, activation='relu')(x)
    x = layers.Dropout(0.4)(x)

    outputs = layers.Dense(num_classes, activation='sigmoid')(x)
    model = models.Model(inputs, outputs)
    return model

global_model = build_lung_attnet()
global_model.summary()


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d (Conv2D)     │ (None, 224, 224,  │        896 │ input_layer[0][0] │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d       │ (None, 112, 112,  │          0 │ conv2d[0][0]      │
│ (MaxPooling2D)      │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_1 (Conv2D)   │ (None, 112, 112,  │     18,496 │ max_pooling2d[0]… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_1     │ (None, 56, 56,    │          0 │ conv2d_1[0][0]    │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_2 (Conv2D)   │ (None, 56, 56,    │     73,856 │ max_pooling2d_1[… │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_2     │ (None, 28, 28,    │          0 │ conv2d_2[0][0]    │
│ (MaxPooling2D)      │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_3 (Conv2D)   │ (None, 28, 28,    │    295,168 │ max_pooling2d_2[… │
│                     │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_3     │ (None, 14, 14,    │          0 │ conv2d_3[0][0]    │
│ (MaxPooling2D)      │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_5 (Conv2D)   │ (None, 14, 14,    │      8,224 │ max_pooling2d_3[… │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_6 (Conv2D)   │ (None, 14, 14,    │      8,224 │ max_pooling2d_3[… │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ reshape (Reshape)   │ (None, 196, 32)   │          0 │ conv2d_5[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ reshape_1 (Reshape) │ (None, 196, 32)   │          0 │ conv2d_6[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lambda_2 (Lambda)   │ (None, 196, 196)  │          0 │ reshape[0][0],    │
│                     │                   │            │ reshape_1[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_7 (Conv2D)   │ (None, 14, 14,    │      8,224 │ max_pooling2d_3[… │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lambda (Lambda)     │ (None, 14, 14, 1) │          0 │ max_pooling2d_3[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lambda_1 (Lambda)   │ (None, 14, 14, 1) │          0 │ max_pooling2d_3[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ softmax (Softmax)   │ (None, 196, 196)  │          0 │ lambda_2[0][0]    │
├─────────────────────┼───────────────────┼────────────┼─────────────────

 Total params: 7,533,476 (28.74 MB)

 Trainable params: 7,533,476 (28.74 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
# =======================
# 4. FEDAVG UTILITIES
# =======================
def get_model_weights(model):
    return model.get_weights()

def set_model_weights(model, weights):
    model.set_weights(weights)

def aggregate_weights(weight_list, sizes):
    total = np.sum(sizes)
    new_weights = []
    for weights in zip(*weight_list):
        weighted = np.zeros_like(weights[0])
        for w, s in zip(weights, sizes):
            weighted += (s / total) * w
        new_weights.append(weighted)
    return new_weights

def compile_model(model, lr):
    opt = tf.keras.optimizers.Nadam(learning_rate=lr)
    model.compile(
        optimizer=opt,
        loss='binary_crossentropy',
        metrics=['accuracy']
    )

def evaluate_global(model, test_gen, batch_size):
    test_gen_tmp = test_datagen.flow_from_directory(
        test_dir,
        target_size=IMG_SIZE,
        batch_size=batch_size,
        class_mode='binary',
        shuffle=False
    )
    loss, acc = model.evaluate(test_gen_tmp, verbose=0)
    return acc


In [ ]:
# =======================
# 5. FEDERATED TRAINING LOOP
# =======================
def federated_training(global_model,
                       client_tf_datasets,
                       client_sizes,
                       lr=1e-3,
                       local_epochs=1,
                       comm_rounds=10,
                       batch_size=32):
    compile_model(global_model, lr)
    history_acc = []

    for r in range(comm_rounds):
        print(f"\n--- Communication round {r+1}/{comm_rounds} ---")
        global_weights = get_model_weights(global_model)
        client_weights = []

        for cid, ds in enumerate(client_tf_datasets):
            print(f"  Client {cid+1}/{len(client_tf_datasets)}")
            client_model = build_lung_attnet()
            compile_model(client_model, lr)
            set_model_weights(client_model, global_weights)
            client_model.fit(ds, epochs=local_epochs, verbose=0)
            client_weights.append(get_model_weights(client_model))
            tf.keras.backend.clear_session()

        new_global_weights = aggregate_weights(client_weights, client_sizes)
        set_model_weights(global_model, new_global_weights)

        acc = evaluate_global(global_model, test_gen, batch_size)
        history_acc.append(acc)
        print(f"  Global test accuracy after round {r+1}: {acc:.4f}")

    return history_acc

# Run with fixed LR first (you can plug PSO later if you want)
final_global_model = build_lung_attnet()
final_history_acc = federated_training(
    final_global_model,
    client_tf_datasets,
    client_sizes,
    lr=1e-3,
    local_epochs=1,
    comm_rounds=COMM_ROUNDS,
    batch_size=32
)

print("Final global accuracies per round:", final_history_acc)
print("Best final accuracy:", max(final_history_acc))


NameError: name 'build_lung_attnet' is not defined

In [ ]:
# =======================
# 6. GRAD-CAM SETUP
# =======================
# Make a test generator for visualization
test_gen_vis = test_datagen.flow_from_directory(
    test_dir,
    target_size=IMG_SIZE,
    batch_size=1,
    class_mode='binary',
    shuffle=False
)

test_images_vis = []
test_labels_vis = []
test_paths_vis = []

for i in range(len(test_gen_vis)):
    x, y = next(test_gen_vis)
    test_images_vis.append(x[0])
    test_labels_vis.append(y[0])
    if hasattr(test_gen_vis, 'filepaths'):
        test_paths_vis.append(test_gen_vis.filepaths[i])
    else:
        test_paths_vis.append(test_gen_vis.filenames[i])

test_images_vis = np.array(test_images_vis)
test_labels_vis = np.array(test_labels_vis)
print("Loaded test images:", test_images_vis.shape)

# Find last conv layer
last_conv_layer_name = None
for layer in reversed(final_global_model.layers):
    if isinstance(layer, tf.keras.layers.Conv2D):
        last_conv_layer_name = layer.name
        break

print("Using last conv layer:", last_conv_layer_name)

def make_gradcam_heatmap(img_array, model, last_conv_layer_name, pred_index=None):
    grad_model = tf.keras.models.Model(
        [model.inputs],
        [model.get_layer(last_conv_layer_name).output, model.output]
    )

    with tf.GradientTape() as tape:
        conv_outputs, predictions = grad_model(img_array)
        if pred_index is None:
            pred_index = tf.argmax(predictions[0])
        class_channel = predictions[:, pred_index]

    grads = tape.gradient(class_channel, conv_outputs)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))

    conv_outputs = conv_outputs[0]
    heatmap = conv_outputs @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)
    heatmap = tf.maximum(heatmap, 0) / (tf.reduce_max(heatmap) + 1e-8)
    return heatmap.numpy()

def show_gradcam_on_image(orig_img, heatmap, alpha=0.4):
    img = np.uint8(255 * orig_img)
    heatmap_resized = cv2.resize(heatmap, (img.shape[1], img.shape[0]))
    heatmap_resized = np.uint8(255 * heatmap_resized)
    heatmap_color = cv2.applyColorMap(heatmap_resized, cv2.COLORMAP_JET)
    superimposed_img = cv2.addWeighted(img, 1-alpha, heatmap_color, alpha, 0)
    return superimposed_img

def display_gradcam_example(idx):
    img = test_images_vis[idx]
    inp = np.expand_dims(img, axis=0)
    preds = final_global_model.predict(inp, verbose=0)[0][0]
    label = int(test_labels_vis[idx])
    heatmap = make_gradcam_heatmap(inp, final_global_model, last_conv_layer_name)
    cam_img = show_gradcam_on_image(img, heatmap)

    plt.figure(figsize=(10,4))
    plt.subplot(1,3,1)
    plt.title(f"Original\nTrue: {label}")
    plt.axis('off')
    plt.imshow(img)

    plt.subplot(1,3,2)
    plt.title(f"Grad-CAM\nPred: {preds:.3f}")
    plt.axis('off')
    plt.imshow(heatmap, cmap='jet')

    plt.subplot(1,3,3)
    plt.title("Overlay")
    plt.axis('off')
    plt.imshow(cam_img)
    plt.show()

# Example: show Grad-CAM for first 3 test images
for i in range(3):
    display_gradcam_example(i)


NameError: name 'test_datagen' is not defined

In [ ]:
# =======================
# 7. LIME EXPLANATIONS
# =======================
def lime_predict(images):
    preds = final_global_model.predict(images, verbose=0)
    return np.concatenate([1-preds, preds], axis=1)  # [non-nodule, nodule]

explainer = lime_image.LimeImageExplainer()

def explain_with_lime(idx, positive_label=1):
    img = test_images_vis[idx]
    label = int(test_labels_vis[idx])

    explanation = explainer.explain_instance(
        np.uint8(img*255),
        classifier_fn=lime_predict,
        top_labels=2,
        hide_color=0,
        num_samples=1000
    )

    temp, mask = explanation.get_image_and_mask(
        label=positive_label,
        positive_only=True,
        num_features=5,
        hide_rest=False
    )

    plt.figure(figsize=(10,4))
    plt.subplot(1,2,1)
    plt.title(f"Original\nTrue: {label}")
    plt.axis('off')
    plt.imshow(img)

    plt.subplot(1,2,2)
    plt.title("LIME explanation")
    plt.axis('off')
    plt.imshow(mark_boundaries(temp/255.0, mask))
    plt.show()

# Example: LIME for first 3 test images
for i in range(3):
    explain_with_lime(i, positive_label=1)


In [ ]:
best_lr = float(best_pos[0])
best_bs_raw = int(best_pos[1])
best_bs = max(8, min(64, (best_bs_raw // 8) * 8))
print(f"Using best lr={best_lr:.5f}, batch_size={best_bs}")

client_tf_datasets_best = [
    make_tf_dataset(imgs, lbls, best_bs)
    for (imgs, lbls) in client_datasets
]

final_global_model = build_lung_attnet()
final_history_acc = federated_training(
    final_global_model,
    client_tf_datasets_best,
    client_sizes,
    lr=best_lr,
    local_epochs=1,
    comm_rounds=10,     # as you requested
    batch_size=best_bs
)

print("Final global accuracies per round:", final_history_acc)
print("Best final accuracy:", max(final_history_acc))


In [ ]:
# =======================
# 8. COMBINED VIEW (Original + Grad-CAM + LIME)
# =======================
def show_explainability_triplet(idx, class_name_map=None):
    img = test_images_vis[idx]
    label = int(test_labels_vis[idx])
    inp = np.expand_dims(img, axis=0)
    preds = final_global_model.predict(inp, verbose=0)[0][0]

    heatmap = make_gradcam_heatmap(inp, final_global_model, last_conv_layer_name)
    cam_img = show_gradcam_on_image(img, heatmap)

    explanation = explainer.explain_instance(
        np.uint8(img*255),
        classifier_fn=lime_predict,
        top_labels=2,
        hide_color=0,
        num_samples=1000
    )
    temp, mask = explanation.get_image_and_mask(
        label=1,
        positive_only=True,
        num_features=5,
        hide_rest=False
    )
    lime_img = mark_boundaries(temp/255.0, mask)

    true_name = class_name_map.get(label, str(label)) if class_name_map else str(label)

    plt.figure(figsize=(14,4))
    plt.subplot(1,3,1)
    plt.title(f"Original\nTrue: {true_name}")
    plt.axis('off')
    plt.imshow(img)

    plt.subplot(1,3,2)
    plt.title(f"Grad-CAM\nPred(nodule)={preds:.3f}")
    plt.axis('off')
    plt.imshow(cam_img)

    plt.subplot(1,3,3)
    plt.title("LIME")
    plt.axis('off')
    plt.imshow(lime_img)

    plt.show()

class_name_map = {0: 'non_nodule', 1: 'nodule'}
for i in range(3):
    show_explainability_triplet(i, class_name_map)
